In [ ]:
# Imports
import cv2
import os 
import numpy as np
import glob, os, json
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from IPython.display import display, clear_output
from tqdm.notebook import tqdm
from datetime import datetime

In [ ]:
#Find camera
for i in range(3):
    cam_index = i
    capture = cv2.VideoCapture(cam_index)
    if not capture.isOpened():
        print(f"camera with index {cam_index} not found!")
    else:
        print(f"camera with index {cam_index} found!")
        for j in range(5):
            ret, frame = capture.read()
        ret, frame = capture.read()
        if ret and frame is not None and frame.sum() > 0:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            print(f"Resolution: {frame.shape[1]}x{frame.shape[0]}")
            plt.imshow(frame_rgb)
            plt.title(f"Camera {cam_index} - {frame.shape[1]}x{frame.shape[0]}")
            plt.axis('off')
            plt.show()
        else:
            print("could not read frame")
        capture.release()

In [ ]:
# calibration video Recording 
cam_index = 0
capture = cv2.VideoCapture(cam_index)

capture.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
capture.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
frame_width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"Resolution: {int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))}x{int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))}")
print("Warming up camera................")
num_warmup_frames = 30
for i in range(num_warmup_frames):
    ret, frame = capture.read()
fps = 30

output_dir = r"C:\AAKASH\MS_NOTES\THESIS\Material\calibration_webcam\calrecordings_newsetup3"
os.makedirs(output_dir, exist_ok=True)
base_name = "recorded_calvid"
file_extension = ".mp4"

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_file = os.path.join(output_dir, f"{base_name}_{timestamp}{file_extension}")

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_file, fourcc, fps, (frame_width, frame_height))

num_frames_to_record = 5000
print("Recording... Move checkerboard slowly to all corners and different distances!")

for i in tqdm(range(num_frames_to_record), desc="Recording frames"):
    ret, frame = capture.read()
    if ret:
        out.write(frame)
        if i % 30 == 0:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            plt.imshow(frame_rgb)
            plt.title(f"Recording Frame {i}")
            plt.axis('off')
            clear_output(wait=True)  
            display(plt.gcf())       
            plt.close() 
    else:
        print(f"Frame {i} not captured.")
        break

capture.release()
out.release()
print(f"Recording done! Video saved as '{output_file}'")

In [ ]:
# Extract calibration images from video
VIDEO = r"C:\AAKASH\MS_NOTES\THESIS\Material\calibration_webcam\calrecordings_newsetup3\recorded_calvid_2026-01-21_18-26-50.mp4"  
OUTDIR = r"C:\AAKASH\MS_NOTES\THESIS\Material\calibration_webcam\calrecordings_newsetup3\calimgs"
os.makedirs(OUTDIR, exist_ok=True)

CB_W, CB_H = 9, 6  # inner corners
PATTERN = (CB_W, CB_H)

def is_sharp(gray, thresh=100.0):
    return cv2.Laplacian(gray, cv2.CV_64F).var() > thresh

def corners_center(corners):
    c = corners.reshape(-1, 2)
    return c.mean(axis=0)

def mean_disp(c1, c2):
    a = c1.reshape(-1, 2)
    b = c2.reshape(-1, 2)
    return np.linalg.norm(a - b, axis=1).mean()

def board_size_in_image(corners):
    c = corners.reshape(-1, 2)
    width = np.linalg.norm(c[0] - c[CB_W-1])
    height = np.linalg.norm(c[0] - c[(CB_H-1)*CB_W])
    return (width + height) / 2

cap = cv2.VideoCapture(VIDEO)
stable_queue = []
last_kept_center = None
saved = 0
frame_idx = 0

# Track board sizes to ensure variety of distances
saved_sizes = []

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_idx += 1
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    ret_cb, corners = cv2.findChessboardCorners(gray, PATTERN,flags=cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_FAST_CHECK + cv2.CALIB_CB_NORMALIZE_IMAGE)
    
    if not ret_cb:
        stable_queue.clear()
        continue

    corners= cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1),(cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 50, 1e-3))

    if not is_sharp(gray, thresh=100.0):
        continue

    stable_queue.append(corners.copy())
    if len(stable_queue) < 4:
        continue
    if len(stable_queue) > 5:
        stable_queue.pop(0)

    d1 = mean_disp(stable_queue[-1], stable_queue[-2])
    d2 = mean_disp(stable_queue[-2], stable_queue[-3])
    d3 = mean_disp(stable_queue[-3], stable_queue[-4])
    if not (d1 < 1.0 and d2 < 1.0 and d3 < 1.0):
        continue

    ctr = corners_center(corners)
    if last_kept_center is not None:
        if np.linalg.norm(ctr - last_kept_center) < 30:
            continue

    # Check board size for distance diversity
    board_size = board_size_in_image(corners)
    
    out_path = os.path.join(OUTDIR, f"img_{saved:04d}.png")
    cv2.imwrite(out_path, frame)
    saved += 1
    last_kept_center = ctr
    saved_sizes.append(board_size)
    print(f"Saved image {saved}, board size: {board_size:.0f}px")

    if saved >= 40:
        break

cap.release()
print(f"\nSaved {saved} calibration images to {OUTDIR}")
print(f"Board sizes range: {min(saved_sizes):.0f} - {max(saved_sizes):.0f} pixels")

In [ ]:
#Setup calibration parameters
IMG_GLOB = r"C:\AAKASH\MS_NOTES\THESIS\Material\calibration_webcam\calrecordings_newsetup3\calimgs\*.png"
OUT_DIR = r"C:\AAKASH\MS_NOTES\THESIS\Material\calibration_webcam\calrecordings_newsetup3\results"

CB_COLS = 9       # inner corners horizontally
CB_ROWS = 6       # inner corners vertically
SQUARE_MM = 71.0

ALPHA = 0.0
DROP_WORST = 2    # Drop 2 worst images

os.makedirs(OUT_DIR, exist_ok=True)
print("Images found:", len(glob.glob(IMG_GLOB)))

In [ ]:
#Detect corners in all images
images = sorted(glob.glob(IMG_GLOB))
assert len(images) >= 10, "Need at least 10 images for a good calibration."

objp_unit = np.zeros((CB_ROWS * CB_COLS, 3), np.float32)
objp_unit[:, :2] = np.mgrid[0:CB_COLS, 0:CB_ROWS].T.reshape(-1, 2)

objpoints_unit = []
imgpoints = []
used_paths = []
imsize = None

OUT_DIR = Path(OUT_DIR)
DBG_DIR = OUT_DIR / "dbg"
DBG_DIR.mkdir(parents=True, exist_ok=True)

for p in tqdm(images, desc="Processing images", unit="image"):
    img = cv2.imread(p)
    if img is None:
        print("Skip (cannot read):", p)
        continue
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    if imsize is None:
        imsize = gray.shape[::-1]

    ret_cb, corners = cv2.findChessboardCorners(gray, (CB_COLS, CB_ROWS),flags=cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_NORMALIZE_IMAGE)
    if not ret_cb:
        print("Skip (no corners):", os.path.basename(p))
        continue

    corners = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1),(cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 50, 1e-3))

    objpoints_unit.append(objp_unit.copy())
    imgpoints.append(corners)
    used_paths.append(p)

    dbg = img.copy()
    cv2.drawChessboardCorners(dbg, (CB_COLS, CB_ROWS), corners, True)
    cv2.imwrite(str(DBG_DIR / f"dbg_{Path(p).name}"), dbg)

print(f"Used {len(used_paths)} / {len(images)} images")
assert len(used_paths) >= 10, "Too few valid images after detection."

In [ ]:
#calibration
objpoints_mm = [op.copy() for op in objpoints_unit]
for op in objpoints_mm:
    op[:, :2] *= SQUARE_MM

# Initial calibration
rms, K, D, rvecs, tvecs = cv2.calibrateCamera(objpoints_mm, imgpoints, imsize, None, None)
print(f"Initial RMS: {rms:.4f} px")
print("K:\n", K)
print("D:", D.ravel())

# Per-image reprojection errors
def per_image_errors(objpoints, imgpoints, rvecs, tvecs, K, D):
    errs = []
    for i, (rv, tv) in enumerate(zip(rvecs, tvecs)):
        proj, _ = cv2.projectPoints(objpoints[i], rv, tv, K, D)
        e = np.linalg.norm(proj.reshape(-1, 2) - imgpoints[i].reshape(-1, 2), axis=1).mean()
        errs.append(e)
    return np.array(errs, dtype=np.float32)

errs = per_image_errors(objpoints_mm, imgpoints, rvecs, tvecs, K, D)
order = np.argsort(errs)[::-1]
print("\nWorst frames:")
for k in range(min(5, len(order))):
    i = order[k]
    print(f"  {errs[i]:.3f}px  -> {os.path.basename(used_paths[i])}")

# Drop worst and recalibrate
if DROP_WORST > 0:
    keep_idx = sorted(order[DROP_WORST:])
    objpoints_keep = [objpoints_mm[i] for i in keep_idx]
    imgpoints_keep = [imgpoints[i] for i in keep_idx]
    used_paths = [used_paths[i] for i in keep_idx]
    rms, K, D, rvecs, tvecs = cv2.calibrateCamera(objpoints_keep, imgpoints_keep, imsize, None, None)
    errs = per_image_errors(objpoints_keep, imgpoints_keep, rvecs, tvecs, K, D)
    print(f"\nRecalibrated RMS: {rms:.4f} px (kept {len(used_paths)} images)")
    print("K:\n", K)

In [ ]:
#Validate calibration
fx = K[0, 0]
fy = K[1, 1]
cx = K[0, 2]
cy = K[1, 2]

print("CALIBRATION VALIDATION:")

print(f"Image size: {imsize}")
print(f"fx = {fx:.1f}, fy = {fy:.1f}")
print(f"cx = {cx:.1f}, cy = {cy:.1f}")
print(f"Expected cx ≈ {imsize[0]/2:.1f}, cy ≈ {imsize[1]/2:.1f}")

# FOV calculation
fov_h = 2 * np.arctan(imsize[0] / 2 / fx) * 180 / np.pi
fov_v = 2 * np.arctan(imsize[1] / 2 / fy) * 180 / np.pi
print(f"\nEstimated FOV: {fov_h:.1f}° horizontal, {fov_v:.1f}° vertical")

# Sanity check
print("\n SANITY CHECK:")
if 45 < fov_h < 80:
    print(f" FOV looks reasonable for a webcam ({fov_h:.1f}°)")
else:
    print(f" WARNING: FOV = {fov_h:.1f}° seems unusual!")
    print(f"  Typical webcam FOV is xx-xx°")
    print(f"  Check your SQUARE_MM value!")

if abs(cx - imsize[0]/2) < 30 and abs(cy - imsize[1]/2) < 30:
    print(f" Principal point is near image center")
else:
    print(f" WARNING: Principal point is far from center!")

if rms < 0.5:
    print(f" RMS error is excellent ({rms:.3f} px)")
elif rms < 1.0:
    print(f" RMS error is good ({rms:.3f} px)")
else:
    print(f" WARNING: RMS error is high ({rms:.3f} px)")


In [ ]:
# Save calibration
calib_json = OUT_DIR / "pinhole_calib.json"
with open(calib_json, "w") as f:
    json.dump({
        "model": "pinhole",
        "rms": float(rms),
        "K": K.tolist(),
        "D": D.ravel().tolist(),
        "image_size": list(imsize),
        "images_used": [Path(p).name for p in used_paths]
    }, f, indent=2)
print("Wrote", calib_json)

# Save K and dist as .npy files
np.save(OUT_DIR / "K.npy", K)
np.save(OUT_DIR / "dist.npy", D)
print("Saved K.npy, dist.npy")

# Undistortion maps
newK, roi = cv2.getOptimalNewCameraMatrix(K, D, imsize, alpha=ALPHA)
map1, map2 = cv2.initUndistortRectifyMap(K, D, None, newK, imsize, cv2.CV_16SC2)
np.save(OUT_DIR / "newK.npy", newK)
np.save(OUT_DIR / "map1.npy", map1)
np.save(OUT_DIR / "map2.npy", map2)
print("Saved newK.npy, map1.npy, map2.npy")


print("\nCALIBRATION COMPLETE:")
print(newK)


In [ ]:
MAPS_DIR     = r"C:\AAKASH\MS_NOTES\THESIS\Material\calibration_webcam\calrecordings_newsetup3\results"
DIS_IMG_GLOB = r"C:\AAKASH\MS_NOTES\THESIS\Material\calibration_webcam\calrecordings_newsetup3\calimgs\*.png"

OUT_DIR = os.path.join(MAPS_DIR, "undis_imgs")
os.makedirs(OUT_DIR, exist_ok=True)


#load maps
map1 = np.load(os.path.join(MAPS_DIR, "map1.npy"))
map2 = np.load(os.path.join(MAPS_DIR, "map2.npy"))

dis_images = sorted(glob.glob(DIS_IMG_GLOB))
assert dis_images, f"No images match: {DIS_IMG_GLOB}"


for p in tqdm(dis_images, desc="Undistorting", unit="image"):
           
      assert os.path.exists(p), f"Image not found: {p}"
      img_bgr = cv2.imread(p)
      if img_bgr is None:
            print("Skipping unreadable file:", p)
            continue

      #undistort
      und_bgr = cv2.remap(img_bgr, map1, map2, cv2.INTER_LINEAR)


      base = os.path.splitext(os.path.basename(p))[0]
      out_und  = os.path.join(OUT_DIR, f"{base}_undist.png")
      out_side = os.path.join(OUT_DIR, f"{base}_sidebyside.png")
      cv2.imwrite(out_und, und_bgr)

      side = np.hstack([img_bgr, und_bgr])
      cv2.imwrite(out_side, side)


      img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
      und_rgb = cv2.cvtColor(und_bgr, cv2.COLOR_BGR2RGB)
      plt.figure(figsize=(12,5))
      plt.subplot(1,2,1); plt.imshow(img_rgb); plt.title(f"Original: {os.path.basename(p)}"); plt.axis("off")
      plt.subplot(1,2,2); plt.imshow(und_rgb); plt.title("Undistorted"); plt.axis("off")

      clear_output(wait=True)  
      display(plt.gcf())       
      plt.close()  


In [ ]:
MAPS_DIR = Path(r"C:\AAKASH\MS_NOTES\THESIS\Material\calibration_webcam\calrecordings_newsetup3\results")
VIDEO_IN  = Path(r"C:\AAKASH\MS_NOTES\THESIS\Material\calibration_webcam\calrecordings_newsetup3\recorded_calvid_2026-01-21_18-26-50.mp4")
VIDEO_OUT = MAPS_DIR / "undist_clip.mp4"

map1 = np.load(MAPS_DIR / "map1.npy")
map2 = np.load(MAPS_DIR / "map2.npy")

cap = cv2.VideoCapture(str(VIDEO_IN))
ok, frame = cap.read()
assert ok, f"Cannot read {VIDEO_IN}"
h, w = frame.shape[:2]
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(str(VIDEO_OUT), fourcc, fps, (w, h))

i = 0
while ok:
    und = cv2.remap(frame, map1, map2, cv2.INTER_LINEAR)
    out.write(und)
    ok, frame = cap.read()
    i += 1

cap.release(); out.release()
print(f"Wrote {VIDEO_OUT} ({i} frames @ {fps:.1f} fps)")
